### Humm

In [121]:
(define graph-schema
    '((directed . 0) (weighted . 1) (direct-loop . 2) (body . 3))
)

(define (getp g p)
    (list-ref g (cdr (assq p graph-schema)))
)

(define (replace-by-i i value l)
    (if (= i 0)
        (cons value (cdr l))
        (cons (car l) (replace-by-i (- i 1) value (cdr l)))
    )
)

(define (setp g p value)
    (replace-by-i (cdr (assq p graph-schema)) value g)
)

(define (add-vertex grafo v)
    (let ((body (getp grafo 'body)))
        (if (assq v body)
            grafo
            (setp grafo 'body (cons `(,v . ()) body))
        )
    )
)

(define (replace-by-key key value l)
    (cond
        ((null? l)
            l)
        ((eq? (car (car l)) key)
            (cons `(,key . ,value) (cdr l)))
        (else (cons (car l)
            (replace-by-key key value (cdr l))))
    )
)

; apenas adiciona v2 na lista de adjacências de v1, sem se preocupar com o tipo do grafo
; e sem checar a existência de v1 e v2 antes (dá erro se v1 não existir)
(define (add-direct-edge grafo v1 v2)
    (let ((body (getp grafo 'body)))
        (let ((v1-edges (cdr (assq v1 body))))
            (if (memq v2 v1-edges)
                grafo
                (setp grafo 'body (replace-by-key v1 (cons v2 v1-edges) body))
            )
        )
    )
)

(define (add-edge grafo v1 v2)
    (let ((body (getp grafo 'body)))
        (if (assq v1 body)
            (if (assq v2 body)
                (if (getp grafo 'directed)
                    (add-direct-edge grafo v1 v2)
                    (add-direct-edge (add-direct-edge grafo v1 v2) v2 v1)
                )
                (add-edge (add-vertex grafo v2) v1 v2)
            )
            (add-edge (add-vertex grafo v1) v1 v2)
        )
    )
)

; (directed  weighted  direct-loop  body)
(let ((GRAFO '(#f #f #f ())))

    ; inicialmente planejava usar macros aqui, mas estou usando o interpretador Calysto Scheme que não suporta macros da maneira que vimos em sala
    (define (add-vertex! V)
        (set! GRAFO (add-vertex GRAFO V))
    )
    (define (add-edge! V1 V2)
        (set! GRAFO (add-edge GRAFO V1 V2))
    )

    (display (getp GRAFO 'body))
    (newline)
    (add-vertex! 'B)
    (display (getp GRAFO 'body))
    (newline)
    (add-vertex! 'M)
    (display (getp GRAFO 'body))

    (newline)
    (add-edge! 'B 'A)
    (display (getp GRAFO 'body))

    (newline)
    (add-edge! 'B 'M)
    (display (getp GRAFO 'body))

)

()
((B))
((M) (B))
((A B) (M) (B A))
((A B) (M B) (B M A))